## **Proyecto de Machine Learning: nomadismo digital y coste de vida global**

**Enfoque estratégico:** Clustering de ciudades como destino nómada digital & Regresión para predecir el score nómada de países en función de las variables. 

**Autores:** Daniela Aguirre, Agustín Arganín y Juan F. Cía

**Base del proyecto:** [EDA Nomadismo Digital](https://github.com/juanfcia/EDA-Nomadismo-Digital)

**Datasets y registros:**
1. **Registros:** 4,742 ciudades y 85 países. 
2. **Datasets:** [Variables agregadas](https://github.com/danielaaguirrej55-source/Proyecto-ML---Machine-Learning/blob/proj_ml_juanfcia/src/data_sample/cost-of-living-variables-agregadas.csv), [Mapa de variables](https://github.com/danielaaguirrej55-source/Proyecto-ML---Machine-Learning/blob/proj_ml_juanfcia/src/data_sample/cost-of-living-vars-map.csv), [Digital Nomad Index Circleloop](https://github.com/danielaaguirrej55-source/Proyecto-ML---Machine-Learning/blob/proj_ml_juanfcia/src/data_sample/digital-nomad-index-cicleloop-clean.csv) y [Digital Nomad Index Movingto](https://github.com/danielaaguirrej55-source/Proyecto-ML---Machine-Learning/blob/proj_ml_juanfcia/src/data_sample/digital-nomad-index-movingto-clean.csv).  
3. **Hipótesis validadas en el proyecto del EDA:** 9 hipótesis validadas

### **Problema de negocio**

#### **Contexto**

El nomadismo digital es un fenómeno creciente: profesionales que trabajan en remoto desde cualquier lugar del mundo. Elegir destino implica evaluar decenas de factores — coste de vida, conectividad, seguridad, calidad de vida — que interactúan de forma no lineal.

#### **Preguntas de negocio**

1. ¿Qué factores de coste de vida, conectividad y bienestar determinan el atractivo de un destino para nómadas digitales? 
2. ¿Podemos segmentar las ciudades del mundo en perfiles nómadas y predecir el score de un país?*

#### **Hallazgos del EDA que motivan este proyecto**

| Hallazgo | Estadístico | Implicación ML |
|---|---|---|
| **Conectividad** = predictor #1 del score nómada | r=0.73, p<0.001 | Feature de alta importancia |
| **Bienestar social** correlaciona fuertemente | r=0.70, p<0.001 | Feature de alta importancia |O
| **Los mejores destinos son más caros** | Top 20% coste mayor | Relación no lineal con el score |
| **Alimentación** pesa más que vivienda | Hipótesis rechada | Validar con feature importance |
| **Seguridad no tiene correlación fuerte** | r=0.18, p>0.05 | Variable de baja importancia |
| **Outliers value for money** identificados | Bajo coste + alto score | ¿Forman un clúster propio? |

#### **Enfoque estratético del proyecto**

Pipeline híbrido en 3 fases:

1. **Fase 1 — Clustering (aprendizaje no supervisado):** Segmentar 4,742 ciudades en perfiles nómadas usando 7 índices de coste de vida.
2. **Fase 2 — Regresión (aprendizaje supervisado):** Predecir el `digital_nomad_score` de 81 países usando indicadores de coste, conectividad, bienestar y los perfiles de cluster descubiertos en la fase 1.
3. **Fase 3 - Iteación añadiendo nuevos datos y ver variabilidad predictora:** Incluir datos de series temporales sobre inflación, ingresos por turismo, precio de la vivienda, clima... y analizar si los modelos baseline de fases 1 y 2 modifican su comportamiento para negocio. 

### **Carga de librerías y de datasets iniciales para fases 1 y 2**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Sklearn - Preprocesado
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.feature_selection import SelectKBest, f_regression

# Sklearn - Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

# Sklearn - Regresión
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# Guardado del modelo
import joblib

import warnings
warnings.filterwarnings('ignore')

In [12]:
# Carga de datasets limpios del EDA

df_cities = pd.read_csv('../data_sample/cost-of-living-variables-agregadas.csv')
df_circleloop = pd.read_csv('../data_sample/digital-nomad-index-cicleloop-clean.csv')
df_movingto = pd.read_csv('../data_sample/digital-nomad-index-movingto-clean.csv')

In [17]:
# Número de registros y variables por cada uno de los datasets

print(f'Dataset 1: Cost of Living -> {df_cities.shape[0]:,} ciudades y {df_cities.shape[1]} variables')
print(f'Dataset 2: Circleloop -> {df_circleloop.shape[0]} países y {df_circleloop.shape[1]} variables')
print(f'Dataset 3: Movingto -> {df_movingto.shape[0]} países y {df_movingto.shape[1]} variables')

Dataset 1: Cost of Living -> 4,742 ciudades y 65 variables
Dataset 2: Circleloop -> 85 países y 10 variables
Dataset 3: Movingto -> 40 países y 10 variables


In [15]:
# Imprimimos los tres datasets seguidos para echarle un vistazo

display(df_cities.head(10))
display(df_circleloop.head(10))
display(df_movingto.head(10))

,city_name,country_name,meal_inexpensive_restaurant,meal_midrange_restaurant_2p,mcmeal_fastfood,beer_domestic_restaurant_0_5l,beer_imported_restaurant_0_33l,cappuccino_restaurant,soda_restaurant_0_33l,water_restaurant_0_33l,...,mortgage_interest_rate_20y,data_quality_flag,continent,nomad_housing_cost,basic_basket_index,daily_meal_cost,monthly_nomad_cost,local_purchasing_power,cappuccino_index,housing_salary_ratio
0,Seoul,South Korea,7.68,53.78,6.15,3.07,4.99,3.93,1.48,0.79,...,3.47,1,Asia,650.030,4.631667,11.61,1209.990,2.222845,37.934560,24.168098
1,Shanghai,China,5.69,39.86,5.69,1.14,4.27,3.98,0.53,0.33,...,5.03,1,Asia,830.905,1.995000,9.67,1147.345,1.237527,38.445808,58.519794
2,Guangzhou,China,4.13,28.47,4.98,0.85,1.71,3.54,0.44,0.33,...,5.19,1,Asia,425.365,1.572500,7.67,692.370,1.750047,33.946830,35.105391
3,Mumbai,India,3.68,18.42,3.68,2.46,4.30,2.48,0.48,0.19,...,7.96,1,Asia,408.225,1.078333,6.16,590.785,1.084675,23.108384,63.704530
4,Delhi,India,4.91,22.11,4.30,1.84,3.68,1.77,0.49,0.19,...,8.06,1,Asia,182.575,1.047500,6.68,391.890,1.496491,15.848671,31.131705
5,Dhaka,Bangladesh,1.95,11.71,4.88,5.85,5.12,1.95,0.29,0.16,...,9.26,1,Asia,114.940,1.193333,3.90,286.730,0.979074,17.689162,40.943255
6,Osaka,Japan,7.45,48.39,5.36,3.35,3.72,3.28,1.09,0.81,...,1.49,1,Asia,525.550,3.302500,10.73,1035.305,2.243262,31.288344,22.629023
7,Jakarta,Indonesia,2.59,22.69,3.57,2.06,3.24,2.23,0.61,0.27,...,9.05,1,Asia,391.510,1.765000,4.82,643.550,0.791112,20.552147,76.899356
8,Shenzhen,China,4.27,28.47,4.98,1.14,3.99,4.20,0.47,0.34,...,4.99,1,Asia,586.910,1.829167,8.47,890.265,1.766013,40.695297,37.330017
9,Kinshasa,Congo,15.11,42.63,10.08,1.74,2.50,4.35,2.78,0.84,...,19.33,0,Africa,1362.500,4.215000,19.46,2664.840,0.150103,42.229039,340.625000


,rank,country,broadband_speed_mbps,mobile_speed_mbps,broadband_cost,monthly_rent,happiness_index,migrant_population_pct,remote_jobs_searches,digital_nomad_score
0,1,Canada,149.35,84.54,41.17,1045.0,7.23,21.3,83900,74.35
1,2,UK,76.49,41.72,42.18,1019.7,7.17,14.1,68400,63.43
2,3,Romania,188.55,41.48,9.63,357.5,6.12,2.4,10980,62.28
3,4,Sweden,158.73,56.64,46.77,973.5,7.35,20.0,3490,61.54
4,5,Denmark,179.81,66.68,56.96,1164.9,7.65,12.5,1080,61.49
5,6,France,177.93,50.45,32.84,833.8,6.66,12.8,5360,60.80
6,7,Netherlands,125.82,88.13,50.11,1334.3,7.45,13.4,3440,60.27
7,8,Australia,58.52,88.35,57.11,1262.8,7.22,30.0,17600,60.16
8,9,Switzerland,186.40,73.85,85.15,1699.5,7.56,29.9,3840,60.15
9,10,Germany,120.13,49.67,33.95,906.4,7.08,15.7,12720,60.00


,rank,country,overall_score,internet_speed,cost_of_living,safety,visa_ease,quality_of_life,taxes,tax_free_period
0,1,Portugal,92,90,85,95,95,95,NHR 20%,10 years
1,2,Estonia,91,95,75,90,98,92,0-20%,183 days/year
2,3,Georgia,90,85,88,82,100,80,1%,183 days/year
3,4,Spain,89,88,80,92,92,94,24%,183 days/year
4,5,Thailand,88,85,95,80,90,88,0-35%,183 days/year
5,6,Mexico,87,82,90,75,94,86,1.92-35%,183 days/year
6,7,Czech Republic,86,87,78,88,85,90,15%,183 days/year
7,8,Malaysia,85,80,92,85,90,85,0-30%,182 days/year
8,9,Croatia,84,84,76,88,88,89,24%,1 year
9,10,Costa Rica,83,79,85,80,90,88,0-25%,183 days/year
